# Hazard-Aware Rover Path Planning with Double Deep Q-Learning

AE 696, San Diego State University. Notebook version of the project code, organized like Lab 3.

**What this notebook does:** builds the rover environment, defines the Double DQN pieces one at a time (Q network, replay buffer, epsilon-greedy, Double DQN target, optimization step), trains on randomly generated maps, evaluates on unseen maps, and plots the results.

| Part | Topic |
|:---:|---|
| 0 | Setup |
| A | Environment and maps |
| B | Q network |
| C | Replay buffer |
| D | Epsilon greedy action selection |
| E | Double DQN target |
| F | One optimization step |
| G | Train and evaluate |
| H | Plots and trajectories |

> **Known issue (read first).** With the README's settings the rover only receives a real reward when it reaches the goal, and a random rover almost never does. Training therefore may show `success = 0` throughout. This notebook demonstrates the pipeline; it does not yet include a fix for the sparse reward.

## Part 0 Setup

On Kaggle or Colab, run the install cell once. Locally, use `pip install -e ".[dev]"` from the repo root instead.

In [ ]:
# Run once on Kaggle/Colab (internet must be on). Skip if you installed the repo locally.
# !git clone https://github.com/BrendanUribe/Hazard-Aware-Rover-Path-Planning-with-Deep-Q-Learning.git
# %cd Hazard-Aware-Rover-Path-Planning-with-Deep-Q-Learning
# !git checkout notebook
# %pip install -q gymnasium
# %pip install -q -e . --no-deps

In [ ]:
import json, math, random
from collections import deque, namedtuple

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from rover import RoverEnv
from rover.viz import draw_map

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| device:", device)

## Part A Environment and maps

State: terrain map $M$ (10x10), rover position $(x,y)$, battery $b$. Actions: North, South, East, West.
Rewards: -1 per step, +100 goal, -100 crater, -100 battery depleted, -15 drive into a boulder.
Gravel: 30% chance of slipping to a perpendicular cell, which costs 3 battery instead of 1.

The observation is a flat vector: one-hot terrain (6 x 100) + one-hot rover position (100) + battery / b_max.

In [ ]:
env = RoverEnv()                       # new random map on every reset
obs, _ = env.reset(seed=SEED)
STATE_DIM = env.observation_space.shape[0]
ACTION_DIM = env.action_space.n
print("Observation vector length:", STATE_DIM, "| actions:", ACTION_DIM)
print(env.render())

In [ ]:
# Random-policy rovers on three maps (the lower-bound baseline)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, ax in enumerate(axes):
    e = RoverEnv()
    o, _ = e.reset(seed=SEED + i)
    path, done = [e.pos], False
    while not done:
        o, r, term, trunc, info = e.step(random.randrange(ACTION_DIM))
        path.append(e.pos); done = term or trunc
    draw_map(e.grid, path, ax, f"seed {SEED + i}: {info['outcome']} ({len(path) - 1} steps)")
plt.tight_layout(); plt.show()

## Part B Q network

Maps a batch of states to one Q value per action: Linear, ReLU, Linear, ReLU, Linear. No activation after the last layer.

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),
        )

    def forward(self, x):
        return self.net(x)

q = QNetwork(STATE_DIM, ACTION_DIM).to(device)
print("single:", q(torch.zeros(1, STATE_DIM, device=device)).shape,
      "| batch:", q(torch.zeros(7, STATE_DIM, device=device)).shape)

## Part C Replay buffer

Stores $(s, a, r, s', d)$. `d` is true termination only; a time-limit truncation ends the rollout but keeps `d = 0` so the target still bootstraps.

In [ ]:
Transition = namedtuple("Transition", ["state", "action", "reward", "next_state", "terminated"])

class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, terminated):
        self.buffer.append(Transition(state, action, reward, next_state, terminated))

    def sample(self, batch_size):
        t = random.sample(self.buffer, batch_size)
        return (np.array([x.state for x in t]), np.array([x.action for x in t]),
                np.array([x.reward for x in t]), np.array([x.next_state for x in t]),
                np.array([x.terminated for x in t]))

    def __len__(self):
        return len(self.buffer)

# quick check: capacity keeps the newest items
b = ReplayBuffer(5)
for i in range(7):
    b.push(np.full(STATE_DIM, i, dtype=np.float32), 0, 1.0, np.zeros(STATE_DIM, dtype=np.float32), False)
print("length:", len(b), "| oldest state value kept:", int(b.buffer[0].state[0]))

## Part D Epsilon greedy action selection

With probability $\epsilon$ pick a random action, otherwise $a = \arg\max_a Q_\theta(s,a)$, computed under `torch.no_grad()`.

In [ ]:
def select_action(state, q_net, epsilon, action_dim, device):
    if random.random() < epsilon:
        return random.randint(0, action_dim - 1)
    with torch.no_grad():
        s = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        return int(q_net(s).argmax(dim=1).item())

## Part E Double DQN target

Vanilla DQN:

$$y = r + \gamma (1-d)\,\max_{a'} Q_{\theta^-}(s', a')$$

Double DQN picks the action with the **online** network and scores it with the **target** network:

$$y = r + \gamma (1-d)\,Q_{\theta^-}\big(s', \arg\max_{a'} Q_{\theta}(s', a')\big)$$

`double=True` is Double DQN; `double=False` is vanilla DQN for comparison.

In [ ]:
def compute_targets(rewards, next_states, terminated, q_net, target_net, gamma, double=True):
    rewards = torch.as_tensor(rewards, dtype=torch.float32, device=device).reshape(-1, 1)
    terminated = torch.as_tensor(terminated, dtype=torch.float32, device=device).reshape(-1, 1)
    next_states = torch.as_tensor(next_states, dtype=torch.float32, device=device)
    with torch.no_grad():
        if double:
            best_a = q_net(next_states).argmax(dim=1, keepdim=True)       # select: online
            next_q = target_net(next_states).gather(1, best_a)            # evaluate: target
        else:
            next_q = target_net(next_states).max(dim=1, keepdim=True).values
        return rewards + gamma * (1.0 - terminated) * next_q

# Numerical check where the two targets differ
class Fixed(nn.Module):
    def __init__(self, v): super().__init__(); self.v = torch.tensor(v)
    def forward(self, x): return self.v.to(x.device).repeat(x.shape[0], 1)

online, target = Fixed([[5.0, 1.0]]), Fixed([[2.0, 4.0]])
args = ([1.0], np.zeros((1, STATE_DIM), dtype=np.float32), [0.0], online, target, 0.99)
print("double target :", compute_targets(*args, double=True).item(), "(online picks action 0 -> target says 2.0 -> 1 + 0.99*2.0)")
print("vanilla target:", compute_targets(*args, double=False).item(), "(max of target = 4.0 -> 1 + 0.99*4.0)")

## Part F One optimization step

Sample a minibatch, gather $Q_\theta(s_i, a_i)$, build the targets, Huber loss, update only the online network, clip the gradient norm to 10.

In [ ]:
def optimize_model(q_net, target_net, buffer, optimizer, batch_size, gamma, device, double=True):
    if len(buffer) < batch_size:
        return None
    states, actions, rewards, next_states, terminated = buffer.sample(batch_size)
    states = torch.as_tensor(states, dtype=torch.float32, device=device)
    actions = torch.as_tensor(actions, dtype=torch.int64, device=device).reshape(-1, 1)
    current_q = q_net(states).gather(1, actions)
    targets = compute_targets(rewards, next_states, terminated, q_net, target_net, gamma, double)
    loss = F.smooth_l1_loss(current_q, targets)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(q_net.parameters(), 10.0)
    optimizer.step()
    return loss.item()

## Part G Train and evaluate

Training uses a fresh random map every episode. Evaluation is greedy ($\epsilon = 0$) on maps the agent has not trained on: `n_maps` maps, `episodes_per_map` missions each (slip is still random).

Change `NUM_EPISODES` below. 5,000 episodes is a first real test; 20,000 matches the README protocol.

In [ ]:
NUM_EPISODES = 2000
EVAL_EVERY = 250
DOUBLE = True          # False -> vanilla DQN
P_SLIP, B0 = 0.30, 15

In [ ]:
def evaluate(q_net, n_maps=5, episodes_per_map=20, seed=10_000):
    outcomes, steps, returns = [], [], []
    for m in range(n_maps):
        probe = RoverEnv(p_slip=P_SLIP, b0=B0)
        probe.reset(seed=seed + m)
        env = RoverEnv(fixed_map=probe.grid.copy(), p_slip=P_SLIP, b0=B0)
        for e in range(episodes_per_map):
            s, _ = env.reset(seed=seed + 1000 * m + e)
            done, ret, t = False, 0.0, 0
            while not done:
                s, r, term, trunc, info = env.step(select_action(s, q_net, 0.0, ACTION_DIM, device))
                ret, t, done = ret + r, t + 1, term or trunc
            outcomes.append(info["outcome"]); steps.append(t); returns.append(ret)
    n = len(outcomes)
    return {k: outcomes.count(k) / n for k in ("success", "crater", "battery", "timeout")} | {
        "mean_steps": float(np.mean(steps)), "mean_return": float(np.mean(returns))}


def train_ddqn(seed=SEED, num_episodes=NUM_EPISODES, eval_every=EVAL_EVERY, double=DOUBLE):
    set_seed(seed)
    env = RoverEnv(p_slip=P_SLIP, b0=B0)
    online = QNetwork(STATE_DIM, ACTION_DIM).to(device)
    target = QNetwork(STATE_DIM, ACTION_DIM).to(device)
    target.load_state_dict(online.state_dict()); target.eval()
    buffer = ReplayBuffer(100_000)
    opt = optim.Adam(online.parameters(), lr=1e-3)

    gamma, batch_size, warmup, target_update = 0.99, 64, 1_000, 500
    eps_start, eps_end, eps_decay_episodes = 1.0, 0.05, num_episodes // 4

    returns, losses, evals, updates = [], [], [], 0
    for ep in range(1, num_episodes + 1):
        epsilon = max(eps_end, eps_start - (eps_start - eps_end) * ep / eps_decay_episodes)
        state, _ = env.reset(seed=seed + ep)
        done, ep_ret = False, 0.0
        while not done:
            action = select_action(state, online, epsilon, ACTION_DIM, device)
            next_state, reward, terminated, truncated, _ = env.step(action)
            buffer.push(state, action, reward, next_state, float(terminated))  # truncation is not terminal
            state, ep_ret, done = next_state, ep_ret + reward, terminated or truncated
            if len(buffer) >= warmup:
                loss = optimize_model(online, target, buffer, opt, batch_size, gamma, device, double)
                if loss is not None:
                    losses.append(loss); updates += 1
                    if updates % target_update == 0:
                        target.load_state_dict(online.state_dict())
        returns.append(ep_ret)
        if ep % eval_every == 0:
            m = evaluate(online) | {"episode": ep, "train_return": float(np.mean(returns[-eval_every:])), "eps": epsilon}
            evals.append(m)
            print(f"ep {ep:5d} | success {m['success']:.2f} crater {m['crater']:.2f} battery {m['battery']:.2f} "
                  f"| test return {m['mean_return']:7.1f} | train return {m['train_return']:7.1f} | eps {epsilon:.2f}")
    return online, returns, losses, evals


Q_NET, EPISODE_RETURNS, LOSSES, EVALS = train_ddqn()

## Part H Plots and trajectories

Learning curves from the evaluation history, then the trained rover on maps it has not seen.

In [ ]:
ep = [r["episode"] for r in EVALS]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
for k, c in (("success", "green"), ("crater", "black"), ("battery", "orange"), ("timeout", "gray")):
    a1.plot(ep, [r[k] for r in EVALS], "-o", color=c, label=k)
a1.set(xlabel="training episode", ylabel="fraction of test missions", ylim=(-0.02, 1.02), title="Outcomes on unseen maps")
a1.legend()
a2.plot(ep, [r["mean_return"] for r in EVALS], "-o", label="test return")
a2.plot(ep, [r["train_return"] for r in EVALS], "-o", label="train return")
a2.set(xlabel="training episode", title="Average return"); a2.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Trained (greedy) rover on three unseen maps
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, ax in enumerate(axes):
    e = RoverEnv(p_slip=P_SLIP, b0=B0)
    s, _ = e.reset(seed=90_000 + i)
    path, done = [e.pos], False
    while not done:
        s, r, term, trunc, info = e.step(select_action(s, Q_NET, 0.0, ACTION_DIM, device))
        path.append(e.pos); done = term or trunc
    draw_map(e.grid, path, ax, f"seed {90_000 + i}: {info['outcome']} ({len(path) - 1} steps, battery {e.battery})")
plt.tight_layout(); plt.show()

In [ ]:
# Save results (overwrites if you reuse the same name; change RUN_NAME to keep runs apart)
from pathlib import Path
RUN_NAME = "ddqn_seed42"
out = Path("runs") / RUN_NAME; out.mkdir(parents=True, exist_ok=True)
(out / "log.json").write_text(json.dumps(EVALS, indent=1))
torch.save(Q_NET.state_dict(), out / "model.pt")
print("saved to", out)

## Next steps

- Compare `DOUBLE = False` (vanilla DQN) against `DOUBLE = True` with the same seed.
- Run seeds 0, 1, 2 and report mean and standard deviation, as in the README protocol.
- Fix the sparse reward (for example expert demonstrations in the replay buffer, a curriculum on map size or battery, or a distance-based reward) before expecting `success > 0`.